In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score


DATA_PATH = os.path.join('..', 'vr_legibility_train.csv')

In [2]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv(DATA_PATH)[["yaw", "pitch", "size", "label"]]
print(df.head())

# 특성(X)과 타겟(y) 분리
X = df[["yaw", "pitch", "size"]]
y = df['label']

# 학습/테스트 데이터 분할 (StratifiedKFold 적용)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

     yaw  pitch   size  label
0 -47.30  25.70  201.0      0
1   2.11  18.17  122.9      1
2  45.19  42.13   10.6      0
3 -33.54   0.72    6.5      0
4  18.72   4.42   17.3      1


In [3]:
# 2. 파이프라인 (스케일링 + 모델) 구축
# ---------------------------------------------------------
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced', random_state=42)) # 차후 수정 필요 : 현재 비율 3.5:6.5, 불균형 해소를 위해 balanced 적용 
])

In [4]:
# 3. 커널 탐색 및 하이퍼파라미터 튜닝 (BayesSearchCV)
# ---------------------------------------------------------
kernel_configs = [
    # 공간 1: RBF 커널 전용 (C, gamma)
    ({
        'svm__kernel': Categorical(['rbf']),
        'svm__C': Real(0.1, 1000.0, prior='log-uniform'),
        'svm__gamma': Real(1e-3, 10.0, prior='log-uniform')
    }, 40),
    
    # 공간 2: Linear 커널 전용 (C)
    ({
        'svm__kernel': Categorical(['linear']),
        'svm__C': Real(1e-3, 10.0, prior='log-uniform')
    }, 20),
    
    # 공간 3: Poly 커널 전용 (C, gamma, degree)
    ({
        'svm__kernel': Categorical(['poly']),
        'svm__C': Real(0.1, 1.0, prior='log-uniform'),
        'svm__gamma': Real(1e-3, 1.0, prior='log-uniform'),
        'svm__degree': Integer(2, 3) # 2차식부터 3차식까지 탐색
    }, 50)
]

In [ ]:
# 4. 교차 검증 객체 생성 및 BayesSearchCV 수행
import warnings
warnings.filterwarnings('ignore') # 경고 메시지 무시 

f1_scores = []
acc_scores = []
best_params_per_fold = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
    print(f"\n--- Outer Fold {fold_idx + 1} / 5 ---")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    best_score_inner, best_model_inner, best_params_inner =-1, None, None

    for space, n_iter in kernel_configs:
        bayes_search = BayesSearchCV(
            estimator= pipeline,
            search_spaces= [space],
            n_iter= n_iter,      # 각 커널별 탐색 횟수
            cv= 5,               # 5-Fold 교차 검증
            scoring= 'f1',       
            n_jobs= -1,          # 모든 CPU 코어 사용
            random_state= 42
        )
        bayes_search.fit(X_train, y_train)
        if bayes_search.best_score_ > best_score_inner:
            best_score_inner = bayes_search.best_score_
            best_model_inner = bayes_search.best_estimator_
            best_params_inner = bayes_search.best_params_
        print(f"Completed BayesSearchCV for kernel: {space['svm__kernel'].categories[0]} with best F1: {bayes_search.best_score_:.4f}")
    
    print("-" * 40)
    print(f"Best Params: {best_params_inner}")
    print(f"Best CV F1:  {best_score_inner:.4f}")

    y_pred = best_model_inner.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    f1_scores.append(f1)
    acc_scores.append(acc)
    best_params_per_fold.append(best_params_inner)

print("\n-" * 10 + " FINAL RESULTS " + "-" * 10)
print(f"최종 5-Fold 평균 F1 Score: {np.mean(f1_scores):.4f} (± {np.std(f1_scores):.4f})")
print(f"최종 5-Fold 평균 Accuracy: {np.mean(acc_scores):.4f} (± {np.std(acc_scores):.4f})")


--- Outer Fold 1 / 5 ---
Completed BayesSearchCV for kernel: rbf with best F1: 0.8114
Completed BayesSearchCV for kernel: linear with best F1: 0.5187
Completed BayesSearchCV for kernel: poly with best F1: 0.7155
----------------------------------------
Best Params: OrderedDict({'svm__C': 0.552744134687643, 'svm__gamma': 0.7501993093989322, 'svm__kernel': 'rbf'})
Best CV F1:  0.8114

--- Outer Fold 2 / 5 ---
Completed BayesSearchCV for kernel: rbf with best F1: 0.7989
Completed BayesSearchCV for kernel: linear with best F1: 0.5151
Completed BayesSearchCV for kernel: poly with best F1: 0.6852
----------------------------------------
Best Params: OrderedDict({'svm__C': 0.378657638506354, 'svm__gamma': 0.8231100195347548, 'svm__kernel': 'rbf'})
Best CV F1:  0.7989

--- Outer Fold 3 / 5 ---
Completed BayesSearchCV for kernel: rbf with best F1: 0.8018
Completed BayesSearchCV for kernel: linear with best F1: 0.5676
Completed BayesSearchCV for kernel: poly with best F1: 0.7003
---------------

In [ ]:
# 5. 최종 모델 학습 및 저장
import joblib

print(" [최종 모델 학습 시작]")
print("=========================================")

best_final_score = -1
final_model = None
final_params = None

# 전체 데이터 사용하여 최종 최적화 수행
for space, n_iter in kernel_configs:
    bayes_search_final = BayesSearchCV(
        estimator=pipeline,
        search_spaces=[space],
        n_iter=n_iter,       
        cv=5,                
        scoring='f1_macro',  
        n_jobs=-1,           
        random_state=42
    )
   
    bayes_search_final.fit(X, y)
    
    if bayes_search_final.best_score_ > best_final_score:
        best_final_score = bayes_search_final.best_score_
        final_model = bayes_search_final.best_estimator_
        final_params = bayes_search_final.best_params_
        
    print(f"Completed BayesSearchCV for kernel: {space['svm__kernel'].categories[0]} with best F1: {bayes_search_final.best_score_:.4f}")
    

print("\n [최종 모델 파라미터]")
print(final_params)

# 학습이 완료된 final_model 객체를 하드디스크에 파일로 저장합니다.
joblib.dump(final_model, os.path.join('..', 'vr_legibility_svm_model.pkl'))
print("✅ 최종 모델이 'vr_legibility_svm_model.pkl' 파일로 저장되었습니다!")

 [최종 모델 학습 시작]
Completed BayesSearchCV for kernel: rbf with best F1: 0.8449
Completed BayesSearchCV for kernel: linear with best F1: 0.5803
Completed BayesSearchCV for kernel: poly with best F1: 0.7426

 [최종 배포용 모델 파라미터]
OrderedDict({'svm__C': 0.32328717544504154, 'svm__gamma': 1.186404590595178, 'svm__kernel': 'rbf'})
✅ 최종 모델이 'vr_legibility_svm_model.pkl' 파일로 저장되었습니다!


In [2]:
# 6. 모델 배포 후 예측 (새로운 데이터에 대한 예측)
import joblib
import os
import pandas as pd

# 1. 저장해둔 모델 불러오기 
deployed_model = joblib.load(os.path.join('..', 'vr_legibility_svm_model.pkl'))

# 2. 새 데이터가 불러오기

new_data = pd.DataFrame({
    'height': [180.0, 160.0],
    'x_coord': [1250.5, -450.2],
    'y_coord': [45.2, 210.8]
})

#new_data = pd.read_csv(os.path.join('..', 'vr_legibility_test.csv')) 

# 3. 새로운 데이터 예측
predictions = deployed_model.predict(new_data)
print("새로운 데이터 예측 결과 (0 또는 1):", predictions)

새로운 데이터 예측 결과 (0 또는 1): [0 1]
